# PyPowSyBl LF Four-Configuration Playground

Use this notebook to run AC load flow on the PGLib cases with four controlled OpenLoadFlow configurations:

1. `strict_pm_like`: single slack, no reactive limits, no operational controls.
2. `q_limits_only`: single slack, reactive limits/PV-PQ switching enabled, no operational controls.
3. `ops_only`: single slack, operational controls enabled, reactive limits disabled.
4. `ops_and_q_limits`: single slack, operational controls enabled, reactive limits/PV-PQ switching enabled.

The notebook can also apply the replacement-note variants we have been using: Original, VS init, Gen P/Q, Gen P/Q + VS, and Gen P/Q + VG.

In [29]:
from __future__ import annotations

import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        has_canonical_script = (path / 'scripts' / 'pypowsybl_replacement_experiment_matrix.py').exists()
        has_pglib_data = (path / 'data' / 'raw' / 'pglib' / 'matpower_mat').exists()
        if has_canonical_script and has_pglib_data:
            return path
    raise RuntimeError('Could not find project root from current working directory.')


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from pypowsybl_replacement_experiment_matrix import (  # noqa: E402
    VARIANT_LABELS,
    _case_paths,
    apply_replacement_variant,
    get_table,
    load_case_network,
    make_replacement_variants,
    parse_replacement_notes,
    require_pypowsybl,
    run_openloadflow,
)

pp = require_pypowsybl()
lf = pp.loadflow

print('Project root:', PROJECT_ROOT)
print('PyPowSyBl:', getattr(pp, '__version__', 'unknown'))

Project root: c:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning
PyPowSyBl: 1.15.0


## Settings

Edit this cell first. Keep `DISTRIBUTED_SLACK = False` if you want the strict single-slack comparison closest to PowerModels `compute_ac_pf`.

In [30]:
# Data folder: choose 'base_case', 'api', or 'sad'.
SCENARIO_FOLDER = 'base_case'

# Cases we discussed. Start small before running all large systems.
CASE_NAMES = [
    'pglib_opf_case300_ieee',
    'pglib_opf_case1888_rte',
    'pglib_opf_case6470_rte',
    'pglib_opf_case1354_pegase',
    'pglib_opf_case9241_pegase',
    'pglib_opf_case13659_pegase',
    'pglib_opf_case24464_goc',
    # 'pglib_opf_case78484_epigrids',  # very large; uncomment only when you really want it
]

# Variants from the replacement-note experiments.
VARIANT_NAMES = [
    'original',
    'BV_from_warmstart',
    'G_PGQG_from',
    'G_PGQG_from_plus_BV_from',
    'G_PGQG_from_plus_GVG_from',
]

# Four requested LF configurations.
CONFIG_NAMES = [
    'strict_pm_like',
    'q_limits_only',
    'ops_only',
    'ops_and_q_limits',
]

# Initialization modes to run in the batch. Use ['DC_VALUES'] to add only the DC-angle initialization layer.
# Use ['UNIFORM_VALUES', 'DC_VALUES'] if you want to regenerate both layers from scratch.
VOLTAGE_INIT_MODES = ['DC_VALUES']

# Keep False for the closest PowerModels-style comparison.
DISTRIBUTED_SLACK = False

# For distributed slack experiments later, set DISTRIBUTED_SLACK=True and choose a balance type.
DISTRIBUTED_BALANCE_TYPE = lf.BalanceType.PROPORTIONAL_TO_GENERATION_P_MAX

PROVIDER = 'OpenLoadFlow'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'interactive' / 'pypowsybl_lf_four_config_playground'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MAT_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pglib' / 'matpower_mat' / SCENARIO_FOLDER
BASE_M_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pglib' / 'matpower_m' / SCENARIO_FOLDER

print('MAT dir:', BASE_MAT_DIR)
print('M dir:', BASE_M_DIR)
print('Output dir:', OUTPUT_DIR)

MAT dir: c:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\data\raw\pglib\matpower_mat\base_case
M dir: c:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\data\raw\pglib\matpower_m\base_case
Output dir: c:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\results\interactive\pypowsybl_lf_four_config_playground


In [31]:
@dataclass(frozen=True)
class LFConfig:
    name: str
    label: str
    use_reactive_limits: bool
    operational_controls: bool


CONFIGS = {
    'strict_pm_like': LFConfig(
        name='strict_pm_like',
        label='Single slack | Q limits off | Ops off',
        use_reactive_limits=False,
        operational_controls=False,
    ),
    'q_limits_only': LFConfig(
        name='q_limits_only',
        label='Single slack | Q limits on | Ops off',
        use_reactive_limits=True,
        operational_controls=False,
    ),
    'ops_only': LFConfig(
        name='ops_only',
        label='Single slack | Q limits off | Ops on',
        use_reactive_limits=False,
        operational_controls=True,
    ),
    'ops_and_q_limits': LFConfig(
        name='ops_and_q_limits',
        label='Single slack | Q limits on | Ops on',
        use_reactive_limits=True,
        operational_controls=True,
    ),
}


def enum_name(value: Any) -> str | None:
    if value is None:
        return None
    return str(getattr(value, 'name', value)).split('.')[-1]


def voltage_init_enum(voltage_init_mode_name: str):
    try:
        return getattr(lf.VoltageInitMode, voltage_init_mode_name)
    except AttributeError as exc:
        available = [name for name in dir(lf.VoltageInitMode) if name.isupper()]
        raise ValueError(f'Unknown voltage init mode {voltage_init_mode_name!r}. Available: {available}') from exc


def make_lf_parameters(config: LFConfig, voltage_init_mode_name: str = 'UNIFORM_VALUES'):
    provider_parameters = {
        'acSolverType': 'NEWTON_RAPHSON',
        'maxNewtonRaphsonIterations': '1000',
        'reactiveLimitsMaxPqPvSwitch': '3',
        'slackDistributionFailureBehavior': 'FAIL',
        'useActiveLimits': 'true',
        'disableInconsistentVoltageControls': 'false',
        'transformerVoltageControlMode': 'INCREMENTAL_VOLTAGE_CONTROL',
        'shuntVoltageControlMode': 'WITH_GENERATOR_VOLTAGE_CONTROL',
        'phaseShifterControlMode': 'CONTINUOUS_WITH_DISCRETISATION',
    }
    balance_type = DISTRIBUTED_BALANCE_TYPE if DISTRIBUTED_SLACK else None
    voltage_init_mode = voltage_init_enum(voltage_init_mode_name)
    params = lf.Parameters(
        voltage_init_mode=voltage_init_mode,
        distributed_slack=bool(DISTRIBUTED_SLACK),
        balance_type=balance_type,
        use_reactive_limits=bool(config.use_reactive_limits),
        transformer_voltage_control_on=bool(config.operational_controls),
        shunt_compensator_voltage_control_on=bool(config.operational_controls),
        phase_shifter_regulation_on=bool(config.operational_controls),
        read_slack_bus=True,
        write_slack_bus=True,
        component_mode=lf.ComponentMode.MAIN_CONNECTED,
        provider_parameters=provider_parameters,
    )
    settings = {
        'voltage_init_mode': voltage_init_mode_name,
        'distributed_slack': bool(DISTRIBUTED_SLACK),
        'balance_type': enum_name(balance_type),
        'use_reactive_limits': bool(config.use_reactive_limits),
        'operational_controls': bool(config.operational_controls),
        'transformer_voltage_control_on': bool(config.operational_controls),
        'shunt_compensator_voltage_control_on': bool(config.operational_controls),
        'phase_shifter_regulation_on': bool(config.operational_controls),
        'read_slack_bus': True,
        'write_slack_bus': True,
        'component_mode': 'MAIN_CONNECTED',
        'provider_parameters': provider_parameters,
    }
    return params, settings


def is_converged(status: Any) -> bool:
    return str(status).split('.')[-1].upper() == 'CONVERGED'


def numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors='coerce')

## Small summary helpers

These collect the evidence you said you want to inspect: convergence, iterations, slack, voltage range, generator P/Q, Q-limit pressure, branch loading, and whether controls have usable metadata.

In [32]:
def balance_summary(network) -> dict[str, float]:
    generators = get_table(network, 'get_generators')
    loads = get_table(network, 'get_loads')
    gen_col = 'target_p' if 'target_p' in generators else 'p'
    load_col = 'p0' if 'p0' in loads else 'p'
    total_gen = float(numeric(generators.get(gen_col, pd.Series(dtype=float))).fillna(0.0).sum())
    total_load = float(numeric(loads.get(load_col, pd.Series(dtype=float))).fillna(0.0).sum())
    return {'total_generation_mw': total_gen, 'total_load_mw': total_load, 'gen_minus_load_mw': total_gen - total_load}


def voltage_summary(network) -> dict[str, Any]:
    buses = get_table(network, 'get_buses')
    out = {'bus_voltage_min': math.nan, 'bus_voltage_max': math.nan, 'bus_angle_min': math.nan, 'bus_angle_max': math.nan}
    if buses.empty:
        return out
    if 'v_mag' in buses:
        v = numeric(buses['v_mag'])
        out['bus_voltage_min'] = float(v.min())
        out['bus_voltage_max'] = float(v.max())
    if 'v_angle' in buses:
        a = numeric(buses['v_angle'])
        out['bus_angle_min'] = float(a.min())
        out['bus_angle_max'] = float(a.max())
    return out


def generator_summary(network) -> dict[str, Any]:
    gens = get_table(network, 'get_generators')
    out = {
        'n_generators': len(gens),
        'gen_p_sum_mw': math.nan,
        'gen_q_sum_mvar': math.nan,
        'n_generators_q_below_min': math.nan,
        'n_generators_q_above_max': math.nan,
        'n_generators_q_at_limit': math.nan,
        'n_generators_voltage_regulator_on': math.nan,
    }
    if gens.empty:
        return out
    if 'p' in gens:
        out['gen_p_sum_mw'] = float(numeric(gens['p']).fillna(0.0).sum())
    elif 'target_p' in gens:
        out['gen_p_sum_mw'] = float(numeric(gens['target_p']).fillna(0.0).sum())
    if 'q' in gens:
        out['gen_q_sum_mvar'] = float(numeric(gens['q']).fillna(0.0).sum())
    elif 'target_q' in gens:
        out['gen_q_sum_mvar'] = float(numeric(gens['target_q']).fillna(0.0).sum())
    if 'voltage_regulator_on' in gens:
        out['n_generators_voltage_regulator_on'] = int(gens['voltage_regulator_on'].fillna(False).astype(bool).sum())
    if {'q', 'min_q', 'max_q'}.issubset(gens.columns):
        q = numeric(gens['q'])
        qmin = numeric(gens['min_q'])
        qmax = numeric(gens['max_q'])
        known = q.notna() & qmin.notna() & qmax.notna()
        eps = 1e-4
        out['n_generators_q_below_min'] = int((known & (q < qmin - eps)).sum())
        out['n_generators_q_above_max'] = int((known & (q > qmax + eps)).sum())
        out['n_generators_q_at_limit'] = int((known & ((q - qmin).abs().le(eps) | (q - qmax).abs().le(eps))).sum())
    return out


def branch_loading_summary(network) -> dict[str, Any]:
    branches = get_table(network, 'get_branches')
    limits = get_table(network, 'get_operational_limits')
    out = {'n_branches': len(branches), 'n_operational_limits': len(limits), 'max_branch_s_mva': math.nan, 'max_branch_loading_pct': math.nan}
    if branches.empty:
        return out
    for col in ('p1', 'q1', 'p2', 'q2'):
        if col in branches:
            branches[col] = numeric(branches[col])
    s_values = []
    if {'p1', 'q1'}.issubset(branches.columns):
        s_values.append((branches['p1'].pow(2) + branches['q1'].pow(2)).pow(0.5))
    if {'p2', 'q2'}.issubset(branches.columns):
        s_values.append((branches['p2'].pow(2) + branches['q2'].pow(2)).pow(0.5))
    if s_values:
        s = pd.concat(s_values, axis=1).max(axis=1)
        out['max_branch_s_mva'] = float(s.max())
    return out


def operational_control_support(network) -> dict[str, Any]:
    ratio = get_table(network, 'get_ratio_tap_changers')
    phase = get_table(network, 'get_phase_tap_changers')
    shunts = get_table(network, 'get_shunt_compensators')
    svcs = get_table(network, 'get_static_var_compensators')
    out = {
        'n_ratio_tap_changers': len(ratio),
        'n_ratio_tap_changers_regulating': 0,
        'n_ratio_tap_changers_with_steps': 0,
        'n_phase_tap_changers': len(phase),
        'n_phase_tap_changers_regulating': 0,
        'n_phase_tap_changers_with_steps': 0,
        'n_shunts': len(shunts),
        'n_shunts_voltage_regulation_on': 0,
        'n_shunts_with_multiple_sections': 0,
        'n_svcs': len(svcs),
        'n_svcs_regulating': 0,
    }
    if not ratio.empty:
        if 'regulating' in ratio:
            out['n_ratio_tap_changers_regulating'] = int(ratio['regulating'].fillna(False).astype(bool).sum())
        if 'step_count' in ratio:
            out['n_ratio_tap_changers_with_steps'] = int(numeric(ratio['step_count']).gt(1).sum())
    if not phase.empty:
        if 'regulating' in phase:
            out['n_phase_tap_changers_regulating'] = int(phase['regulating'].fillna(False).astype(bool).sum())
        if 'step_count' in phase:
            out['n_phase_tap_changers_with_steps'] = int(numeric(phase['step_count']).gt(1).sum())
    if not shunts.empty:
        if 'voltage_regulation_on' in shunts:
            out['n_shunts_voltage_regulation_on'] = int(shunts['voltage_regulation_on'].fillna(False).astype(bool).sum())
        if 'max_section_count' in shunts:
            out['n_shunts_with_multiple_sections'] = int(numeric(shunts['max_section_count']).gt(1).sum())
    if not svcs.empty and 'regulating' in svcs:
        out['n_svcs_regulating'] = int(svcs['regulating'].fillna(False).astype(bool).sum())
    out['supports_operational_controls'] = int(any(
        out[key] > 0 for key in [
            'n_ratio_tap_changers_regulating', 'n_ratio_tap_changers_with_steps',
            'n_phase_tap_changers_regulating', 'n_phase_tap_changers_with_steps',
            'n_shunts_voltage_regulation_on', 'n_shunts_with_multiple_sections', 'n_svcs_regulating'
        ]
    ))
    return out

## Run one case/variant/config

Use this first before running a batch.

In [33]:
VARIANTS = make_replacement_variants(include_bus_voltage=True)


def load_case_and_apply_variant(case_name: str, variant_name: str):
    mat_path, m_path = _case_paths(case_name, BASE_MAT_DIR, BASE_M_DIR)
    network = load_case_network(mat_path)
    notes = parse_replacement_notes(m_path)
    variant = VARIANTS[variant_name]
    audit, counts = apply_replacement_variant(
        network,
        notes,
        variant,
        case_name=case_name,
        parameter_scenario='notebook_four_config',
        strict=False,
    )
    return network, {'mat_path': mat_path, 'm_path': m_path, 'audit': audit, 'replacement_counts': counts}


def run_one(
    case_name: str,
    variant_name: str,
    config_name: str,
    voltage_init_mode_name: str = 'UNIFORM_VALUES',
    save_states: bool = False,
) -> dict[str, Any]:
    config = CONFIGS[config_name]
    network, meta = load_case_and_apply_variant(case_name, variant_name)
    params, settings = make_lf_parameters(config, voltage_init_mode_name=voltage_init_mode_name)
    before_balance = balance_summary(network)
    support = operational_control_support(network)
    started = time.perf_counter()
    try:
        lf_result = run_openloadflow(network, params, provider=PROVIDER)
        error = ''
    except Exception as exc:
        lf_result = {
            'status': 'EXCEPTION',
            'status_text': str(exc),
            'converged': False,
            'iteration_count': math.nan,
            'slack_bus_id': None,
            'slack_active_power_mismatch': math.nan,
            'distributed_active_power': math.nan,
            'elapsed_seconds': time.perf_counter() - started,
        }
        error = repr(exc)

    row = {
        'scenario_folder': SCENARIO_FOLDER,
        'case_name': case_name,
        'variant_name': variant_name,
        'variant_label': VARIANT_LABELS.get(variant_name, variant_name),
        'config_name': config_name,
        'config_label': config.label,
        'voltage_init_mode': voltage_init_mode_name,
        'converged': bool(lf_result.get('converged', False)),
        'status': lf_result.get('status'),
        'status_text': lf_result.get('status_text'),
        'iteration_count': lf_result.get('iteration_count'),
        'slack_bus_id': lf_result.get('slack_bus_id'),
        'slack_active_power_mismatch_mw': lf_result.get('slack_active_power_mismatch'),
        'distributed_active_power_mw': lf_result.get('distributed_active_power'),
        'elapsed_seconds': lf_result.get('elapsed_seconds'),
        'error': error,
        **settings,
        **{f'before_{k}': v for k, v in before_balance.items()},
        **{f'after_{k}': v for k, v in balance_summary(network).items()},
        **voltage_summary(network),
        **generator_summary(network),
        **branch_loading_summary(network),
        **support,
        **meta['replacement_counts'],
    }

    if save_states:
        stem = f"{case_name}__{variant_name}__{config_name}__{voltage_init_mode_name}"
        get_table(network, 'get_buses').to_csv(OUTPUT_DIR / f'{stem}__buses.csv', index=False)
        get_table(network, 'get_generators').to_csv(OUTPUT_DIR / f'{stem}__generators.csv', index=False)
        get_table(network, 'get_branches').to_csv(OUTPUT_DIR / f'{stem}__branches.csv', index=False)

    return row


# Example single run. Change these three names and rerun this cell.
row = run_one(
    case_name='pglib_opf_case300_ieee',
    variant_name='G_PGQG_from_plus_GVG_from',
    config_name='strict_pm_like',
    voltage_init_mode_name='DC_VALUES',
    save_states=False,
)
pd.Series(row)

scenario_folder                          base_case
case_name                   pglib_opf_case300_ieee
variant_name             G_PGQG_from_plus_GVG_from
variant_label                         Gen P/Q + VG
config_name                         strict_pm_like
                                   ...            
n_applied_gen_qg                                69
n_applied_gen_vg                                69
n_applied_loads                                  0
n_applied_bus_voltage                            0
n_nan_updates_skipped                          207
Length: 66, dtype: object

## Batch run selected cases, variants, and four configurations

This writes a compact CSV to `results/interactive/pypowsybl_lf_four_config_playground`.

In [34]:
rows = []
for voltage_init_mode_name in VOLTAGE_INIT_MODES:
    for case_name in CASE_NAMES:
        for variant_name in VARIANT_NAMES:
            for config_name in CONFIG_NAMES:
                print(f'Running {case_name} | {variant_name} | {config_name} | {voltage_init_mode_name} ...')
                rows.append(
                    run_one(
                        case_name,
                        variant_name,
                        config_name,
                        voltage_init_mode_name=voltage_init_mode_name,
                        save_states=False,
                    )
                )

new_results = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / f'{SCENARIO_FOLDER}_four_config_lf_rows.csv'

if csv_path.exists() and csv_path.stat().st_size:
    old_results = pd.read_csv(csv_path)
    if 'voltage_init_mode' not in old_results.columns:
        old_results['voltage_init_mode'] = 'UNIFORM_VALUES'
    combined = pd.concat([old_results, new_results], ignore_index=True, sort=False)
    key_cols = ['scenario_folder', 'case_name', 'variant_name', 'config_name', 'voltage_init_mode']
    results = combined.drop_duplicates(subset=key_cols, keep='last').reset_index(drop=True)
else:
    results = new_results

results.to_csv(csv_path, index=False)
print('Wrote', csv_path)
results

Running pglib_opf_case300_ieee | original | strict_pm_like | DC_VALUES ...
Running pglib_opf_case300_ieee | original | q_limits_only | DC_VALUES ...
Running pglib_opf_case300_ieee | original | ops_only | DC_VALUES ...
Running pglib_opf_case300_ieee | original | ops_and_q_limits | DC_VALUES ...
Running pglib_opf_case300_ieee | BV_from_warmstart | strict_pm_like | DC_VALUES ...
Running pglib_opf_case300_ieee | BV_from_warmstart | q_limits_only | DC_VALUES ...
Running pglib_opf_case300_ieee | BV_from_warmstart | ops_only | DC_VALUES ...
Running pglib_opf_case300_ieee | BV_from_warmstart | ops_and_q_limits | DC_VALUES ...
Running pglib_opf_case300_ieee | G_PGQG_from | strict_pm_like | DC_VALUES ...
Running pglib_opf_case300_ieee | G_PGQG_from | q_limits_only | DC_VALUES ...
Running pglib_opf_case300_ieee | G_PGQG_from | ops_only | DC_VALUES ...
Running pglib_opf_case300_ieee | G_PGQG_from | ops_and_q_limits | DC_VALUES ...
Running pglib_opf_case300_ieee | G_PGQG_from_plus_BV_from | strict_

,scenario_folder,case_name,variant_name,variant_label,config_name,config_label,converged,status,status_text,iteration_count,...,n_shunts_with_multiple_sections,n_svcs,n_svcs_regulating,supports_operational_controls,n_applied_gen_pg,n_applied_gen_qg,n_applied_gen_vg,n_applied_loads,n_applied_bus_voltage,n_nan_updates_skipped
0,base_case,pglib_opf_case300_ieee,original,Original,strict_pm_like,Single slack | Q limits off | Ops off,False,ComponentStatus.FAILED,Solver failed,530,...,0,0,0,0,0,0,0,0,0,0
1,base_case,pglib_opf_case300_ieee,original,Original,q_limits_only,Single slack | Q limits on | Ops off,False,ComponentStatus.FAILED,Solver failed,530,...,0,0,0,0,0,0,0,0,0,0
2,base_case,pglib_opf_case300_ieee,original,Original,ops_only,Single slack | Q limits off | Ops on,False,ComponentStatus.FAILED,Solver failed,530,...,0,0,0,0,0,0,0,0,0,0
3,base_case,pglib_opf_case300_ieee,original,Original,ops_and_q_limits,Single slack | Q limits on | Ops on,False,ComponentStatus.FAILED,Solver failed,530,...,0,0,0,0,0,0,0,0,0,0
4,base_case,pglib_opf_case300_ieee,BV_from_warmstart,VS init,strict_pm_like,Single slack | Q limits off | Ops off,False,ComponentStatus.FAILED,Solver failed,530,...,0,0,0,0,0,0,0,0,300,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,base_case,pglib_opf_case24464_goc,G_PGQG_from_plus_BV_from,Gen P/Q + VS,ops_and_q_limits,Single slack | Q limits on | Ops on,False,ComponentStatus.FAILED,Solver failed,108,...,0,0,0,0,1591,1591,0,0,24464,0
276,base_case,pglib_opf_case24464_goc,G_PGQG_from_plus_GVG_from,Gen P/Q + VG,strict_pm_like,Single slack | Q limits off | Ops off,False,ComponentStatus.FAILED,Solver failed,117,...,0,0,0,0,1591,1591,1591,0,0,4773
277,base_case,pglib_opf_case24464_goc,G_PGQG_from_plus_GVG_from,Gen P/Q + VG,q_limits_only,Single slack | Q limits on | Ops off,False,ComponentStatus.FAILED,Solver failed,117,...,0,0,0,0,1591,1591,1591,0,0,4773
278,base_case,pglib_opf_case24464_goc,G_PGQG_from_plus_GVG_from,Gen P/Q + VG,ops_only,Single slack | Q limits off | Ops on,False,ComponentStatus.FAILED,Solver failed,117,...,0,0,0,0,1591,1591,1591,0,0,4773


In [36]:
summary = (
    results
    .groupby(['case_name', 'voltage_init_mode', 'config_name'], dropna=False)
    .agg(
        converged_variants=('converged', 'sum'),
        total_variants=('converged', 'size'),
        median_iterations=('iteration_count', 'median'),
        max_q_below_min=('n_generators_q_below_min', 'max'),
        max_q_above_max=('n_generators_q_above_max', 'max'),
        supports_operational_controls=('supports_operational_controls', 'max'),
    )
    .reset_index()
)
summary_path = OUTPUT_DIR / f'{SCENARIO_FOLDER}_four_config_lf_summary.csv'
summary.to_csv(summary_path, index=False)
print('Wrote', summary_path)
summary

Wrote c:\Users\Efthymis\OneDrive - University College Dublin\power-systems-tools-learning\results\interactive\pypowsybl_lf_four_config_playground\base_case_four_config_lf_summary.csv


,case_name,voltage_init_mode,config_name,converged_variants,total_variants,median_iterations,max_q_below_min,max_q_above_max,supports_operational_controls
0,pglib_opf_case1354_pegase,DC_VALUES,ops_and_q_limits,5,5,10.0,129,0,0
1,pglib_opf_case1354_pegase,DC_VALUES,ops_only,5,5,3.0,82,15,0
2,pglib_opf_case1354_pegase,DC_VALUES,q_limits_only,5,5,10.0,129,0,0
3,pglib_opf_case1354_pegase,DC_VALUES,strict_pm_like,5,5,3.0,82,15,0
4,pglib_opf_case1354_pegase,UNIFORM_VALUES,ops_and_q_limits,5,5,10.0,129,0,0
5,pglib_opf_case1354_pegase,UNIFORM_VALUES,ops_only,5,5,3.0,82,15,0
6,pglib_opf_case1354_pegase,UNIFORM_VALUES,q_limits_only,5,5,10.0,129,0,0
7,pglib_opf_case1354_pegase,UNIFORM_VALUES,strict_pm_like,5,5,3.0,82,15,0
8,pglib_opf_case13659_pegase,DC_VALUES,ops_and_q_limits,0,5,187.0,0,0,0
9,pglib_opf_case13659_pegase,DC_VALUES,ops_only,1,5,187.0,1541,0,0


## Quick pivot tables

In [37]:
convergence_pivot = results.pivot_table(
    index=['case_name', 'voltage_init_mode', 'variant_label'],
    columns='config_name',
    values='converged',
    aggfunc='first',
)
convergence_pivot

config_name                                                ops_and_q_limits  \
case_name                 voltage_init_mode variant_label                     
pglib_opf_case1354_pegase DC_VALUES         Gen P/Q                    True   
                                            Gen P/Q + VG               True   
                                            Gen P/Q + VS               True   
                                            Original                   True   
                                            VS init                    True   
...                                                                     ...   
pglib_opf_case9241_pegase UNIFORM_VALUES    Gen P/Q                   False   
                                            Gen P/Q + VG               True   
                                            Gen P/Q + VS              False   
                                            Original                  False   
                                            VS init                   False   

config_name                                                ops_only  \
case_name                 voltage_init_mode variant_label             
pglib_opf_case1354_pegase DC_VALUES         Gen P/Q            True   
                                            Gen P/Q + VG       True   
                                            Gen P/Q + VS       True   
                                            Original           True   
                                            VS init            True   
...                                                             ...   
pglib_opf_case9241_pegase UNIFORM_VALUES    Gen P/Q            True   
                                            Gen P/Q + VG       True   
                                            Gen P/Q + VS       True   
                                            Original           True   
                                            VS init            True   

config_name                                                q_limits_only  \
case_name                 voltage_init_mode variant_label                  
pglib_opf_case1354_pegase DC_VALUES         Gen P/Q                 True   
                                            Gen P/Q + VG            True   
                                            Gen P/Q + VS            True   
                                            Original                True   
                                            VS init                 True   
...                                                                  ...   
pglib_opf_case9241_pegase UNIFORM_VALUES    Gen P/Q                False   
                                            Gen P/Q + VG            True   
                                            Gen P/Q + VS           False   
                                            Original               False   
                                            VS init                False   

config_name                                                strict_pm_like  
case_name                 voltage_init_mode variant_label                  
pglib_opf_case1354_pegase DC_VALUES         Gen P/Q                  True  
                                            Gen P/Q + VG             True  
                                            Gen P/Q + VS             True  
                                            Original                 True  
                                            VS init                  True  
...                                                                   ...  
pglib_opf_case9241_pegase UNIFORM_VALUES    Gen P/Q                  True  
                                            Gen P/Q + VG             True  
                                            Gen P/Q + VS             True  
                                            Original                 True  
                                            VS init                  True  

[70 rows x 4 columns]

In [26]:
iteration_pivot = results.pivot_table(
    index=['case_name', 'voltage_init_mode', 'variant_label'],
    columns='config_name',
    values='iteration_count',
    aggfunc='first',
)
iteration_pivot

config_name                               ops_and_q_limits  ops_only  \
case_name                  variant_label                               
pglib_opf_case1354_pegase  Gen P/Q                      12         4   
                           Gen P/Q + VG                  6         3   
                           Gen P/Q + VS                 12         4   
                           Original                     10         3   
                           VS init                      10         3   
pglib_opf_case13659_pegase Gen P/Q                     189       189   
                           Gen P/Q + VG                 10         9   
                           Gen P/Q + VS                189       189   
                           Original                    150       150   
                           VS init                     150       150   
pglib_opf_case1888_rte     Gen P/Q                     515       445   
                           Gen P/Q + VG                555       631   
                           Gen P/Q + VS                515       445   
                           Original                    407       625   
                           VS init                     407       625   
pglib_opf_case24464_goc    Gen P/Q                      78        78   
                           Gen P/Q + VG                109       109   
                           Gen P/Q + VS                 78        78   
                           Original                     94        94   
                           VS init                      94        94   
pglib_opf_case300_ieee     Gen P/Q                    1005         4   
                           Gen P/Q + VG               1005         4   
                           Gen P/Q + VS               1005         4   
                           Original                    530       530   
                           VS init                     530       530   
pglib_opf_case6470_rte     Gen P/Q                      70       121   
                           Gen P/Q + VG                 37       133   
                           Gen P/Q + VS                 70       121   
                           Original                    133        48   
                           VS init                     133        48   
pglib_opf_case9241_pegase  Gen P/Q                     101         4   
                           Gen P/Q + VG                  8         4   
                           Gen P/Q + VS                101         4   
                           Original                     78         5   
                           VS init                      78         5   

config_name                               q_limits_only  strict_pm_like  
case_name                  variant_label                                 
pglib_opf_case1354_pegase  Gen P/Q                   12               4  
                           Gen P/Q + VG               6               3  
                           Gen P/Q + VS              12               4  
                           Original                  10               3  
                           VS init                   10               3  
pglib_opf_case13659_pegase Gen P/Q                  189             189  
                           Gen P/Q + VG              10               9  
                           Gen P/Q + VS             189             189  
                           Original                 150             150  
                           VS init                  150             150  
pglib_opf_case1888_rte     Gen P/Q                  515             445  
                           Gen P/Q + VG             555             631  
                           Gen P/Q + VS             515             445  
                           Original                 407             625  
                           VS init                  407             625  
pglib_opf_case24464_goc    Gen P/Q                   78              78  
   

## Inspect one solved/failed state in detail

This reloads one case/variant/config and returns the bus, generator, branch, tap, and shunt tables after the LF attempt. Set `save_states=True` to export them.

In [27]:
INSPECT_CASE = 'pglib_opf_case1888_rte'
INSPECT_VARIANT = 'G_PGQG_from_plus_BV_from'
INSPECT_CONFIG = 'strict_pm_like'
INSPECT_VOLTAGE_INIT_MODE = 'DC_VALUES'

_ = run_one(INSPECT_CASE, INSPECT_VARIANT, INSPECT_CONFIG, voltage_init_mode_name=INSPECT_VOLTAGE_INIT_MODE, save_states=True)
network, _meta = load_case_and_apply_variant(INSPECT_CASE, INSPECT_VARIANT)
params, _settings = make_lf_parameters(CONFIGS[INSPECT_CONFIG], voltage_init_mode_name=INSPECT_VOLTAGE_INIT_MODE)
try:
    _result = run_openloadflow(network, params, provider=PROVIDER)
    print(_result)
except Exception as exc:
    print('LF exception:', repr(exc))

buses = get_table(network, 'get_buses')
generators = get_table(network, 'get_generators')
branches = get_table(network, 'get_branches')
ratio_taps = get_table(network, 'get_ratio_tap_changers')
phase_taps = get_table(network, 'get_phase_tap_changers')
shunts = get_table(network, 'get_shunt_compensators')

display(buses.head())
display(generators.head())
display(branches.head())
display(ratio_taps.head())
display(phase_taps.head())
display(shunts.head())

{'status': 'ComponentStatus.FAILED', 'status_text': 'Solver failed', 'converged': False, 'component_count': 1, 'main_component_status': 'ComponentStatus.FAILED', 'all_components_converged': False, 'iteration_count': 445, 'reference_bus_id': 'VL-1311_9', 'slack_bus_id': 'VL-1311_9', 'slack_active_power_mismatch': 2250805.892968554, 'distributed_active_power': 0.0, 'elapsed_seconds': 1.4200011999928392, 'elapsed_seconds_single': 1.4200011999928392, 'component_results':    component                  status    status_text  iteration_count  \
0          0  ComponentStatus.FAILED  Solver failed              445   

  reference_bus_id  distributed_active_power slack_bus_id  \
0        VL-1311_9                       0.0    VL-1311_9   

   slack_active_power_mismatch  
0                 2.250806e+06  , 'raw_results': [ComponentResult(connected_component_num=0, synchronous_component_num=0, status=FAILED, status_text=Solver failed, iteration_count=445, reference_bus_id='VL-1311_9', slack_bus_re

,id,name,v_mag,v_angle,connected_component,synchronous_component,voltage_level_id
0,VL-1_0,,1.031493,-43.492413,0,0,VL-1
1,VL-2_0,,1.052035,-1.699896,0,0,VL-2
2,VL-3_0,,1.044838,-18.267388,0,0,VL-3
3,VL-4_0,,1.064605,-20.592316,0,0,VL-4
4,VL-4_1,,1.060563,-25.285142,0,0,VL-4


,id,name,energy_source,target_p,min_p,max_p,min_q,max_q,rated_s,reactive_limits_kind,target_v,target_q,voltage_regulator_on,regulated_element_id,p,q,i,voltage_level_id,bus_id,connected
0,GEN-14,,OTHER,2.38,5.52,27.6,0.0,0.0,100.0,MIN_MAX,1.016260,0.0,True,GEN-14,NaN,NaN,NaN,VL-14,VL-14_0,True
1,GEN-14#0,,OTHER,0.64,4.80,24.0,0.0,0.0,100.0,MIN_MAX,1.016260,0.0,True,GEN-14#0,NaN,NaN,NaN,VL-14,VL-14_0,True
2,GEN-46,,OTHER,28.00,6.00,30.0,-4.0,10.0,100.0,MIN_MAX,1.004215,-4.0,True,GEN-46,NaN,NaN,NaN,VL-46,VL-46_0,True
3,GEN-58,,OTHER,0.00,1.20,6.0,1.9,2.0,100.0,MIN_MAX,1.004310,0.0,True,GEN-58,NaN,NaN,NaN,VL-58,,False
4,GEN-82,,OTHER,1.30,5.06,25.3,0.0,0.0,100.0,MIN_MAX,1.015620,0.0,True,GEN-82,NaN,NaN,NaN,VL-82,VL-82_0,True


,id,type,voltage_level1_id,bus1_id,connected1,voltage_level2_id,bus2_id,connected2,p1,q1,i1,p2,q2,i2
0,LINE-1833-1,LINE,VL-709,VL-709_0,True,VL-1,VL-1_0,True,NaN,NaN,NaN,NaN,NaN,NaN
1,LINE-29-2,LINE,VL-29,VL-29_0,True,VL-2,VL-2_0,True,NaN,NaN,NaN,NaN,NaN,NaN
2,LINE-675-2,LINE,VL-674,VL-674_1,True,VL-2,VL-2_0,True,NaN,NaN,NaN,NaN,NaN,NaN
3,LINE-1271-2,LINE,VL-1269,VL-1269_2,True,VL-2,VL-2_0,True,NaN,NaN,NaN,NaN,NaN,NaN
4,LINE-30-2,LINE,VL-30,VL-30_0,True,VL-2,VL-2_0,True,NaN,NaN,NaN,NaN,NaN,NaN


,id,side,tap,solved_tap_position,low_tap,high_tap,step_count,oltc,regulating,target_v,target_deadband,regulating_bus_id


,id,side,tap,solved_tap_position,low_tap,high_tap,step_count,oltc,regulating,regulation_mode,regulation_value,target_deadband,regulating_bus_id
0,TWT-154-152,,0,NaN,0,0,1,True,False,CURRENT_LIMITER,NaN,NaN,
1,TWT-430-605,,0,NaN,0,0,1,True,False,CURRENT_LIMITER,NaN,NaN,
2,TWT-431-999,,0,NaN,0,0,1,True,False,CURRENT_LIMITER,NaN,NaN,
3,TWT-1273-1052,,0,NaN,0,0,1,True,False,CURRENT_LIMITER,NaN,NaN,


,id,name,g,b,model_type,max_section_count,section_count,solved_section_count,voltage_regulation_on,target_v,target_deadband,regulating_bus_id,p,q,i,voltage_level_id,bus_id,connected
0,SHUNT-2050,,0.0,-64.0,LINEAR,1,1,NaN,False,NaN,NaN,VL-39_0,NaN,NaN,NaN,VL-39,VL-39_0,True
1,SHUNT-49,,0.0,-80.0,LINEAR,1,1,NaN,False,NaN,NaN,VL-49_0,NaN,NaN,NaN,VL-49,VL-49_0,True
2,SHUNT-72,,0.0,-80.0,LINEAR,1,1,NaN,False,NaN,NaN,VL-72_0,NaN,NaN,NaN,VL-72,VL-72_0,True
3,SHUNT-2051,,0.0,-64.0,LINEAR,1,1,NaN,False,NaN,NaN,VL-120_0,NaN,NaN,NaN,VL-120,VL-120_0,True
4,SHUNT-2019,,0.0,-64.0,LINEAR,1,1,NaN,False,NaN,NaN,VL-167_0,NaN,NaN,NaN,VL-167,VL-167_0,True
